In [1]:
import pandas as pd
import numpy as np
import random

from faker import Faker

fake = Faker("en_IN")

random.seed(42)
np.random.seed(42)
Faker.seed(42)

In [2]:
customers_df = pd.read_csv("Customers1.csv")

accounts_df = pd.read_csv("Accounts.csv")

In [3]:
customers_df.shape, accounts_df.shape

((10000, 14), (15000, 8))

In [4]:
customers_df["DOB"] = pd.to_datetime(customers_df["DOB"])

today = pd.Timestamp.today()

customers_df["Age"] = (
    (today - customers_df["DOB"]).dt.days // 365
)

In [5]:
customers_df[["DOB", "Age"]].head()

,DOB,Age
0,1972-10-13,53
1,1976-10-19,49
2,1988-05-26,38
3,1982-10-10,43
4,1981-07-06,45


In [6]:
eligible_customers = customers_df[
    (customers_df["Age"] >= 21) &
    (customers_df["Annual_Income"] >= 300000)
].copy()

In [7]:
eligible_customers.shape

(8616, 15)

In [8]:
active_customers = accounts_df[
    accounts_df["Status"] == "Active"
]["Customer_ID"].unique()

In [9]:
eligible_customers = eligible_customers[
    eligible_customers["Customer_ID"].isin(active_customers)
]

In [10]:
eligible_customers.shape

(8129, 15)

In [11]:
eligible_customers = eligible_customers.sample(
    n=4000,
    random_state=42
).reset_index(drop=True)

In [12]:
eligible_customers.shape

(4000, 15)

In [13]:
CARD_RULES = {

    "Retail": {

        "types": ["Silver", "Gold"],

        "weights": [70, 30],

        "limit": (30000, 200000)

    },

    "Premium": {

        "types": ["Gold", "Platinum"],

        "weights": [60, 40],

        "limit": (200000, 800000)

    },

    "VIP": {

        "types": ["Platinum"],

        "weights": [100],

        "limit": (800000, 2500000)

    }

}

In [14]:
CARD_STATUS = [

    "Active",
    "Blocked",
    "Expired"

]

CARD_STATUS_WEIGHTS = [

    94,
    3,
    3

]

In [15]:
eligible_customers.shape

(4000, 15)

In [16]:
eligible_customers.head()

,Customer_ID,First_Name,Last_Name,Gender,DOB,Email,Phone,City,State,Occupation,Annual_Income,Marital_Status,Customer_Segment,Join_Date,Age
0,C01625,Yamini,Saini,Female,2000-05-14,yamini.saini706@email.com,8170959212,Hyderabad,Telangana,Accountant,446670,Married,Retail,2025-08-14,26
1,C01930,Siddharth,Jani,Female,1975-04-13,siddharth.jani845@email.com,8665682275,Kolkata,West Bengal,Data Analyst,1163821,Single,Premium,2023-03-28,51
2,C04943,Max,Jha,Male,1961-06-19,max.jha7@email.com,7243257013,Mumbai,Maharashtra,Data Analyst,1402997,Married,Premium,2022-05-12,65
3,C05930,Megha,Deol,Male,1957-12-03,megha.deol815@email.com,6043690653,Hyderabad,Telangana,Data Analyst,1193629,Single,Premium,2020-06-11,68
4,C03963,Onveer,Jani,Male,1957-04-11,onveer.jani905@email.com,7471489504,Hyderabad,Telangana,Doctor,1401120,Single,Premium,2021-07-19,69


In [17]:
credit_cards = []

card_number = 1

In [18]:
for _, customer in eligible_customers.iterrows():

    customer_id = customer["Customer_ID"]

    segment = customer["Customer_Segment"]

    join_date = pd.to_datetime(customer["Join_Date"])

    rules = CARD_RULES[segment]

    card_type = random.choices(
        rules["types"],
        weights=rules["weights"],
        k=1
    )[0]

    credit_limit = random.randint(
        rules["limit"][0],
        rules["limit"][1]
    )

    utilization = random.uniform(0.10, 0.80)

    outstanding_balance = round(
        credit_limit * utilization,
        2
    )

    available_credit = round(
        credit_limit - outstanding_balance,
        2
    )

    card_status = random.choices(
        CARD_STATUS,
        weights=CARD_STATUS_WEIGHTS,
        k=1
    )[0]

    issue_date = fake.date_between(
        start_date=join_date.date(),
        end_date="today"
    )

    expiry_date = pd.to_datetime(issue_date) + pd.DateOffset(years=5)

    card_id = f"CC{card_number:06d}"

    card_number += 1

    credit_cards.append({

        "Card_ID": card_id,

        "Customer_ID": customer_id,

        "Card_Type": card_type,

        "Credit_Limit": credit_limit,

        "Outstanding_Balance": outstanding_balance,

        "Available_Credit": available_credit,

        "Card_Status": card_status,

        "Issue_Date": issue_date,

        "Expiry_Date": expiry_date.date()

    })

In [19]:
credit_cards_df = pd.DataFrame(credit_cards)

credit_cards_df.shape

(4000, 9)

In [20]:
credit_cards_df.head()

,Card_ID,Customer_ID,Card_Type,Credit_Limit,Outstanding_Balance,Available_Credit,Card_Status,Issue_Date,Expiry_Date
0,CC000001,C01625,Silver,36556,22631.28,13924.72,Active,2026-04-19,2031-04-19
1,CC000002,C01930,Gold,307473,176394.08,131078.92,Active,2023-09-16,2028-09-16
2,CC000003,C04943,Gold,642417,78534.11,563882.89,Active,2022-06-19,2027-06-19
3,CC000004,C05930,Gold,227824,112287.97,115536.03,Active,2022-10-13,2027-10-13
4,CC000005,C03963,Platinum,639898,162731.46,477166.54,Active,2023-08-18,2028-08-18


In [21]:
credit_cards_df["Card_ID"].duplicated().sum()

np.int64(0)

In [22]:
credit_cards_df.isnull().sum()

Card_ID                0
Customer_ID            0
Card_Type              0
Credit_Limit           0
Outstanding_Balance    0
Available_Credit       0
Card_Status            0
Issue_Date             0
Expiry_Date            0
dtype: int64

In [23]:
credit_cards_df["Card_Type"].value_counts()

Card_Type
Platinum    1822
Gold        1705
Silver       473
Name: count, dtype: int64

In [24]:
credit_cards_df["Card_Status"].value_counts(normalize=True) * 100

Card_Status
Active     93.500
Expired     3.375
Blocked     3.125
Name: proportion, dtype: float64

In [27]:
(
    credit_cards_df["Available_Credit"] +
    credit_cards_df["Outstanding_Balance"] -
    credit_cards_df["Credit_Limit"]
).describe()

count    4000.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
dtype: float64

In [28]:
(
    (
        credit_cards_df["Available_Credit"] +
        credit_cards_df["Outstanding_Balance"]
    ).round(2)
    ==
    credit_cards_df["Credit_Limit"].round(2)
).all()

np.True_

In [26]:
credit_cards_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Card_ID              4000 non-null   object 
 1   Customer_ID          4000 non-null   object 
 2   Card_Type            4000 non-null   object 
 3   Credit_Limit         4000 non-null   int64  
 4   Outstanding_Balance  4000 non-null   float64
 5   Available_Credit     4000 non-null   float64
 6   Card_Status          4000 non-null   object 
 7   Issue_Date           4000 non-null   object 
 8   Expiry_Date          4000 non-null   object 
dtypes: float64(2), int64(1), object(6)
memory usage: 281.4+ KB


In [29]:
credit_cards_df["Issue_Date"] = pd.to_datetime(credit_cards_df["Issue_Date"])
credit_cards_df["Expiry_Date"] = pd.to_datetime(credit_cards_df["Expiry_Date"])

In [30]:
credit_cards_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Card_ID              4000 non-null   object        
 1   Customer_ID          4000 non-null   object        
 2   Card_Type            4000 non-null   object        
 3   Credit_Limit         4000 non-null   int64         
 4   Outstanding_Balance  4000 non-null   float64       
 5   Available_Credit     4000 non-null   float64       
 6   Card_Status          4000 non-null   object        
 7   Issue_Date           4000 non-null   datetime64[ns]
 8   Expiry_Date          4000 non-null   datetime64[ns]
dtypes: datetime64[ns](2), float64(2), int64(1), object(4)
memory usage: 281.4+ KB


In [31]:
credit_cards_df.to_csv(
    "Credit_Cards.csv",
    index=False
)